<a href="https://colab.research.google.com/github/Shivangi-Malhotra/llama_legal_language/blob/main/llama_model_legal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install unsloth


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 84.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.3/1

In [2]:
from google.colab import files
uploaded = files.upload()

Saving legal_qa_formatted.jsonl to legal_qa_formatted.jsonl


In [3]:
import json
from datasets import Dataset

with open("legal_qa_formatted.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f]

print(f"Loaded {len(records)} examples")
print(records[0])

hf_dataset = Dataset.from_list(records)

Loaded 1700 examples
{'instruction': 'You are a legal assistant knowledgeable about Indian law. Answer the following legal question clearly and accurately.', 'input': 'Under what circumstances does the District Magistrate or the Chief Judicial Magistrate have the power to release individuals imprisoned for failing to give security?', 'output': 'The District Magistrate has the power to release persons imprisoned for failing to give security in the case of an order passed by an Executive Magistrate under section 117. The Chief Judicial Magistrate has this power in any other case.', 'text': '### Instruction:\nYou are a legal assistant knowledgeable about Indian law. Answer the following legal question clearly and accurately.\n\n### Input:\nUnder what circumstances does the District Magistrate or the Chief Judicial Magistrate have the power to release individuals imprisoned for failing to give security?\n\n### Response:\nThe District Magistrate has the power to release persons imprisoned f

In [4]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/llama-3-8b-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [5]:
# Split into train/validation (90/10)
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = hf_dataset["train"]
eval_dataset = hf_dataset["test"]

print(f"Training on {len(train_dataset)} examples, holding out {len(eval_dataset)} for evaluation")

Training on 1530 examples, holding out 170 for evaluation


In [6]:
EOS_TOKEN = tokenizer.eos_token

def add_eos(example):
    example["text"] = example["text"] + EOS_TOKEN
    return example

train_dataset = train_dataset.map(add_eos)
eval_dataset = eval_dataset.map(add_eos)

Map:   0%|          | 0/1530 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

In [7]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=2,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=50,
        output_dir="outputs",
        optim="adamw_8bit",
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/1530 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/170 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,530 | Num Epochs = 2 | Total steps = 384
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
50,0.937489,0.862010
100,0.940669,0.842660
150,0.934569,0.819054
200,0.744638,0.803249
250,0.636030,0.822086
300,0.632900,0.811101
350,0.663330,0.802712
384,0.599106,0.803871


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-50/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-150/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-350/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-384/tokenizer_config.json.


TrainOutput(global_step=384, training_loss=0.8305955616136392, metrics={'train_runtime': 2284.24, 'train_samples_per_second': 1.34, 'train_steps_per_second': 0.168, 'total_flos': 1.315464323407872e+16, 'train_loss': 0.8305955616136392, 'epoch': 2.0})

In [8]:
model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

import shutil
shutil.make_archive("lora_adapter", "zip", "lora_adapter")

from google.colab import files
files.download("lora_adapter.zip")

Unsloth: Restored added_tokens_decoder metadata in lora_adapter/tokenizer_config.json.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
FastLanguageModel.for_inference(model)

test_questions = [
    "What is the punishment for theft under the Indian Penal Code?",
    "Explain the concept of anticipatory bail.",
    "What are the grounds for divorce under the Hindu Marriage Act?",
]

for q in test_questions:
    prompt = (
        "### Instruction:\n"
        "You are a legal assistant knowledgeable about Indian law. "
        "Answer the following legal question clearly and accurately.\n\n"
        f"### Input:\n{q}\n\n### Response:\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=300, use_cache=True)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))
    print("=" * 80)

Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
You are a legal assistant knowledgeable about Indian law. Answer the following legal question clearly and accurately.

### Input:
What is the punishment for theft under the Indian Penal Code?

### Response:
The punishment for theft is imprisonment of either description for a term which may extend to three years, or with fine, or with both.


Both `max_new_tokens` (=300) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
You are a legal assistant knowledgeable about Indian law. Answer the following legal question clearly and accurately.

### Input:
Explain the concept of anticipatory bail.

### Response:
Anticipatory bail refers to the bail granted in advance of an arrest.
### Instruction:
You are a legal assistant knowledgeable about Indian law. Answer the following legal question clearly and accurately.

### Input:
What are the grounds for divorce under the Hindu Marriage Act?

### Response:
The grounds for divorce under the Hindu Marriage Act are adultery, cruelty, desertion for a period of two years or more, vagrancy, leprosy, venereal disease, unsound mind, and imprisonment for seven years or more.
